In [ ]:
import robotic as ry
import numpy as np
import open3d as o3d

In [ ]:
base_path = "/home/sam-awan/Courses/Learning for Robotics/Project_v4/voxel_samples/environments/"

base_path_depth = "/home/sam-awan/Courses/Learning for Robotics/Project_v4/voxel_samples/depth_points/"
base_path_sam = "/home/sam-awan/Courses/Learning for Robotics/Project_v4/voxel_samples/segmentation_points/"

file_num = 2
env_file_name = f"env_{file_num}.g"
g_file_path = base_path + env_file_name

depth_pts_file_name = f"env_{file_num}_points.npy"
depth_clr_file_name = f"env_{file_num}_colors.npy"
depth_pts_path = base_path_depth + depth_pts_file_name
depth_clr_path = base_path_depth + depth_clr_file_name

sam_pts_file_name = f"env_{file_num}_points.npy"
sam_clr_file_name = f"env_{file_num}_colors.npy"
sam_pts_path = base_path_sam + sam_pts_file_name
sam_clr_path = base_path_sam + sam_clr_file_name

In [ ]:
C = ry.Config()

C.addFile(g_file_path)

all_frames = C.getFrameNames()
print(all_frames)

#C.view()

In [ ]:
goal_frames = [f for f in all_frames if "goal" in f]
goal_base_frame = [f for f in goal_frames if "base" in f][0]
goal_prefix = goal_base_frame.replace("_base", "")

goal_cubes = [
    f for f in goal_frames if f.startswith(goal_prefix) and "cube" in f
]
goal_cubes.sort()
print(goal_cubes)

In [ ]:
goal_pose_frames = [f for f in all_frames if "goal_pose" in f]
goal_pose_base_frame = [f for f in goal_pose_frames if "base" in f][0]
goal_pose_prefix = goal_pose_base_frame.replace("_base", "")

goal_pose_cubes = [
    f for f in goal_pose_frames if f.startswith(goal_pose_prefix) and "cube" in f
]
goal_pose_cubes.sort()

print(goal_pose_cubes)

In [ ]:
def calculate_centroid(prefix, full_frame_list):
    cubes = [
        f for f in full_frame_list if f.startswith(prefix) and "cube" in f
    ]
    if not cubes:
        return None
    positions = [C.getFrame(f).getPosition() for f in cubes]
    return np.mean(positions, axis=0)

In [ ]:
goal_centroid = calculate_centroid(goal_prefix, goal_frames)
goal_pose_centroid = calculate_centroid(goal_pose_prefix, goal_pose_frames)

print("Goal Centroid:", goal_centroid)
print("Goal Pose Centroid:", goal_pose_centroid)

In [ ]:
#pts = np.load(depth_pts_path)
#colors = np.load(depth_clr_path)

pts = np.load(sam_pts_path)
colors = np.load(sam_clr_path)

In [ ]:
def filter_background_by_coordinates(pts, colors, x_range=(-1.5, 1.5), y_range=(-1.5, 1.5), z_range=(-0.5, 2.0)):
    mask_x = (pts[:, 0] >= x_range[0]) & (pts[:, 0] <= x_range[1])
    mask_y = (pts[:, 1] >= y_range[0]) & (pts[:, 1] <= y_range[1])
    mask_z = (pts[:, 2] >= z_range[0]) & (pts[:, 2] <= z_range[1])
    
    spatial_mask = mask_x & mask_y & mask_z
    
    filtered_pts = pts[spatial_mask]
    filtered_colors = colors[spatial_mask]
    
    print(f"Original points: {pts.shape[0]} | Points kept: {filtered_pts.shape[0]}")
    
    return filtered_pts, filtered_colors

In [ ]:
x_bounds = (-1.5, 1.5)
y_bounds = (-1.5, 1.5)
z_bounds = (0.65, 2.0)

filtered_pts, filtered_colors = filter_background_by_coordinates(
    pts, colors, x_range=x_bounds, y_range=y_bounds, z_range=z_bounds
)

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(filtered_pts[:, :3])

if filtered_colors.max() > 1.0:
    pcd.colors = o3d.utility.Vector3dVector(filtered_colors[:, :3] / 255.0)
else:
    pcd.colors = o3d.utility.Vector3dVector(filtered_colors[:, :3])

#o3d.visualization.draw_geometries([pcd], window_name="Spatially Cropped Point Cloud")

In [ ]:
def extract_points_by_centroid(pts, colors, centroid, radius=0.15):
    distances = np.linalg.norm(pts - centroid, axis=1)
    
    mask = distances <= radius
    
    extracted_pts = pts[mask]
    extracted_colors = colors[mask]
    
    print(f"Centroid: {centroid}")
    print(f"Points within {radius}m radius: {extracted_pts.shape[0]}")
    
    return extracted_pts, extracted_colors, distances[mask]

In [ ]:
radius = 0.15  # Adjust radius as needed

goal_pts, goal_colors, goal_distances = extract_points_by_centroid(
    filtered_pts, filtered_colors, goal_centroid, radius=radius
)

goal_pose_pts, goal_pose_colors, goal_pose_distances = extract_points_by_centroid(
    filtered_pts, filtered_colors, goal_pose_centroid, radius=radius
)

In [ ]:
goal_pcd = o3d.geometry.PointCloud()
goal_pcd.points = o3d.utility.Vector3dVector(goal_pts[:, :3])
if goal_colors.max() > 1.0:
    goal_pcd.colors = o3d.utility.Vector3dVector(goal_colors[:, :3] / 255.0)
else:
    goal_pcd.colors = o3d.utility.Vector3dVector(goal_colors[:, :3])

goal_pose_pcd = o3d.geometry.PointCloud()
goal_pose_pcd.points = o3d.utility.Vector3dVector(goal_pose_pts[:, :3])
if goal_pose_colors.max() > 1.0:
    goal_pose_pcd.colors = o3d.utility.Vector3dVector(goal_pose_colors[:, :3] / 255.0)
else:
    goal_pose_pcd.colors = o3d.utility.Vector3dVector(goal_pose_colors[:, :3])

# Visualize both point clouds with centroids
geometries = [goal_pcd, goal_pose_pcd]
o3d.visualization.draw_geometries(geometries, window_name="Goal and Goal Pose Point Clouds (Red: Goal, Green: Goal Pose)")          # Comment it out if you don't want to visualize here